# Day 7 · Exercise 4: Map Summaries Over Chunks

**What you'll build:** `map_summaries(chunks: list[str], model: str) -> list[str]` — a function that calls `summarize_chunk()` on every element of a chunk list and returns a list of intermediate summary strings of exactly the same length.

**Why it matters:** The Map step is what turns a split document into a set of intermediate results that a Reduce step can collapse — once you can reliably map a single summarisation call over an arbitrary list, you have the engine that powers every long-document pipeline you will build in this course.

## Your Implementation

In [ ]:
import ollama

SUMMARIZE_SYSTEM_PROMPT = (
    "You are a precise summarization assistant. "
    "The user will send you a passage of text — possibly an excerpt from a "
    "larger document. Summarize it in 2–3 sentences of plain prose. "
    "Do not add headings, bullet points, or information that is not present "
    "in the passage. Use only what is written."
)


def summarize_chunk(chunk: str, model: str) -> str:
    """Summarize a single text chunk by calling the Ollama chat API.

    Args:
        chunk: A single passage of text to summarize.
        model: The Ollama model name to use (e.g. "llama3.2").

    Returns:
        A 2–3 sentence summary string.
    """
    messages = [
        {"role": "system", "content": SUMMARIZE_SYSTEM_PROMPT},
        {"role": "user",   "content": chunk},
    ]
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]


def map_summaries(chunks: list[str], model: str) -> list[str]:
    """Apply summarize_chunk() to every element of chunks and return the results.

    Iterates over chunks in order, calls summarize_chunk() on each one, and
    collects the returned strings into a new list. Prints a progress line for
    each chunk so long-running jobs remain visible.

    Args:
        chunks: A list of text strings produced by chunk_text(). Each string
                must already fit the model's context window.
        model:  The Ollama model name to use for every summarisation call.

    Returns:
        A list of summary strings, one per input chunk, in the same order.
        len(result) == len(chunks) is always true.

    Example:
        >>> summaries = map_summaries(["chunk one text", "chunk two text"], "llama3.2")
        >>> len(summaries)
        2
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

# Stub that records calls without touching Ollama
_CALL_LOG: list[str] = []

def _stub_summarize(chunk: str, model: str) -> str:
    _CALL_LOG.append(chunk)
    return f"Summary of: {chunk[:30]}"


def _run_checks():
    score, total = 0, 4

    # Check 1: map_summaries is defined and is callable
    try:
        assert 'map_summaries' in globals(), 'map_summaries is not defined'
        assert callable(map_summaries), 'map_summaries is not callable'
        print(f'{_PASS} Check 1/{total}: map_summaries is defined and callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # cannot safely run remaining checks

    # Monkey-patch summarize_chunk with the stub so checks run without Ollama
    _orig = summarize_chunk
    globals()['summarize_chunk'] = _stub_summarize
    _CALL_LOG.clear()

    try:
        # Check 2: return value is a list
        chunks = ["The Amazon rainforest spans nine countries.",
                  "Deforestation threatens the forest's tipping point."]
        result = map_summaries(chunks, "llama3.2")
        try:
            assert isinstance(result, list), \
                f'expected list, got {type(result).__name__}'
            print(f'{_PASS} Check 2/{total}: map_summaries returns a list')
            score += 1
        except Exception as e:
            print(f'{_FAIL} Check 2/{total}: {e}')

        # Check 3: output length matches input length
        try:
            assert len(result) == len(chunks), (
                f'expected {len(chunks)} summaries, got {len(result)}'
            )
            print(f'{_PASS} Check 3/{total}: output list has same length as input list ({len(chunks)})')
            score += 1
        except Exception as e:
            print(f'{_FAIL} Check 3/{total}: {e}')

        # Check 4: every element is a non-empty string
        try:
            bad = [
                (i, v) for i, v in enumerate(result)
                if not isinstance(v, str) or not v.strip()
            ]
            assert not bad, (
                f'elements at indices {[i for i, _ in bad]} are empty or not strings'
            )
            print(f'{_PASS} Check 4/{total}: every element is a non-empty string')
            score += 1
        except Exception as e:
            print(f'{_FAIL} Check 4/{total}: {e}')

    except Exception as outer:
        print(f'{_FAIL} Unexpected error during checks: {outer}')
    finally:
        globals()['summarize_chunk'] = _orig

    # Edge-case sub-check (no score penalty): empty list → empty list without error
    try:
        empty_result = map_summaries([], "llama3.2")
        if empty_result == []:
            print(f'{_PASS} Bonus check: map_summaries([]) returns [] (no exception)')
        else:
            print(f'{_FAIL} Bonus check: map_summaries([]) should return [], got {empty_result!r}')
    except Exception as e:
        print(f'{_FAIL} Bonus check: map_summaries([]) raised an exception: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {score}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Right now `map_summaries` calls `summarize_chunk` sequentially — chunk 1 finishes before chunk 2 starts. For a 20-chunk document that means 20 round-trips in series.

Try rewriting the function to fire all calls **concurrently** using `asyncio.gather`:

```python
import asyncio

async def map_summaries_concurrent(chunks: list[str], model: str) -> list[str]:
    tasks = [summarize_chunk(chunk, model) for chunk in chunks]
    return list(await asyncio.gather(*tasks))
```

Time both versions on a 5-chunk input with `%%timeit` and observe the speedup. This foreshadows Day 9, where you will build a fully parallel pipeline with rate limiting to stay within an API's requests-per-minute ceiling.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def map_summaries(chunks: list[str], model: str) -> list[str]:
    """Apply summarize_chunk() to every element of chunks and return the results."""
    summaries = []
    total = len(chunks)
    for i, chunk in enumerate(chunks, start=1):
        print(f"  Summarizing chunk {i}/{total}...")
        summary = summarize_chunk(chunk, model)
        summaries.append(summary)
    return summaries
```

**Why this works:** The function initialises an empty accumulator list and iterates with `enumerate(chunks, start=1)` so the progress line reads "1/N" rather than "0/N-1" — a small detail that makes long-running jobs far easier to follow. Each `summarize_chunk(chunk, model)` call sends one chunk to Ollama and waits for the reply, then appends the result. Because the loop appends in the same order as the input, `len(result) == len(chunks)` is guaranteed. An empty input list causes the loop body to never execute, so `summaries` is returned as `[]` — no special-case code needed.
</details>